# Neural network for predicting velocity field

## Data and modules

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
from torchvision.io import decode_image

import numpy as np
import os
import pandas as pd


In [ ]:
class VelocityDataset(Dataset):
    def __init__(self, frame_pairs, velocity_fields):
        """
        frame_pairs: list of arrays, each (4, H, W)
        velocity_fields: list of arrays, each (2, H, W)
        """
        self.frame_pairs = frame_pairs
        self.velocity_fields = velocity_fields

    def __len__(self):
        return len(self.frame_pairs)

    def __getitem__(self, idx):
        x = torch.tensor(self.frame_pairs[idx], dtype=torch.float32)
        y = torch.tensor(self.velocity_fields[idx], dtype=torch.float32)
        return x, y


## Neural Network

In [ ]:
class Velocity3DNet(nn.Module):
    def __init__(self, in_channels=2, num_frames=2, out_channels=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),

            nn.Conv3d(32, 64, kernel_size=3, stride=(1, 2, 2), padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),

            nn.Conv3d(64, 128, kernel_size=3, stride=(1, 2, 2), padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(128, 64, kernel_size=(1,4,4), stride=(1,2,2), padding=(0,1,1)),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose3d(64, 32, kernel_size=(1,4,4), stride=(1,2,2), padding=(0,1,1)),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),

            nn.Conv3d(32, out_channels, kernel_size=(2,3,3), padding=(0,1,1))
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        x = x.squeeze(2)  # to get the 2-dimensional velocity field
        return x


## Training

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

model = Velocity3DNet(in_channels=2, num_frames=2, out_channels=2).to(device)
criterion = nn.CrossEntropyLoss()  # or nn.L1Loss() or nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
